# Feature Engineering

This notebook creates new features from the cleaned e-commerce dataset to support deeper analysis and modeling.  
Features are grouped by domain: Time, Financial, Discount, Customer, Product, Logistics, and Risk/Behavior.

In [10]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.2f}'.format)

In [11]:
# Load the cleaned dataset
ecommerce_df = pd.read_csv(
    'dataset/cleaned/ecommerce_cleaned.csv',
    parse_dates=['order_date']
)

In [12]:
ecommerce_df.sample(5)

,order_date,order_year,order_month,is_weekend,customer_name,gender,age,customer_segment,country,order_status,category,sub_category,unit_price_usd,quantity,discount_percent,revenue_usd,profit_usd,profit_margin_percent,payment_method,shipping_method,shipping_cost_usd,delivery_days,shipping_country,rating,customer_loyalty_score,coupon_used,session_duration_min,pages_visited,abandoned_cart_before,fraud_risk_score,device_type,campaign_source,traffic_source
654080,2024-08-28 20:10:50.003616,2024,8,No,David Frank,Male,30,Regular,Germany,Completed,Electronics,Audio,15.67,3,10,42.31,18.55,43.84,Apple Pay,Economy,14.51,5,Germany,5,31.90,No,42.80,20,Yes,40.50,Mobile,Instagram,Email
718609,2025-05-02 03:37:15.974806,2025,5,No,Emily Downs,Female,21,Regular,Canada,Completed,Electronics,Laptops,137.90,5,10,620.55,279.45,45.03,Bank Transfer,Standard,15.63,8,Canada,3,86.60,Yes,24.50,7,Yes,62.00,Desktop,Affiliate,Search
209113,2026-01-09 11:05:49.340391,2026,1,No,Jennifer Ellis,Female,72,Regular,Netherlands,Completed,Home,Kitchen,96.80,4,0,387.20,172.12,44.45,PayPal,Economy,17.38,14,Netherlands,5,86.80,Yes,20.80,4,No,31.10,Desktop,Affiliate,Social
227955,2025-12-08 07:37:35.624245,2025,12,No,Elizabeth Parker,Female,67,Regular,Spain,Completed,Home,Decor,257.81,5,5,"1,224.60",404.75,33.05,Debit Card,Standard,9.69,4,Spain,4,92.10,Yes,13.60,16,Yes,30.50,Desktop,Facebook,Direct
25717,2024-03-30 02:03:12.691245,2024,3,Yes,Keith West,Male,67,Regular,Italy,Completed,Electronics,Smartphones,202.01,1,5,191.91,99.32,51.75,Credit Card,Express,17.77,3,Italy,2,3.90,No,44.80,16,No,7.20,Mobile,Organic,Search


## 1. Time Features

In [13]:
# Day of week (Mon, Tue, ...)
ecommerce_df.insert(
    ecommerce_df.columns.get_loc('order_month') + 1,
    'order_day_of_week',
    ecommerce_df['order_date'].dt.strftime('%a')
)

# Part of day
ecommerce_df.insert(
    ecommerce_df.columns.get_loc('order_day_of_week') + 1,
    'part_day',
    pd.cut(
        ecommerce_df['order_date'].dt.hour,
        bins=[0, 6, 12, 18, 24],
        labels=['Night', 'Morning', 'Afternoon', 'Evening'],
        right=False
    )
)

## 2. Financial / Revenue Features

In [14]:
# Gross revenue before discount
ecommerce_df.insert(
    ecommerce_df.columns.get_loc('quantity') + 1,
    'gross_revenue_usd',
    ecommerce_df.eval('quantity * unit_price_usd')
)

# Actual discount amount
ecommerce_df.insert(
    ecommerce_df.columns.get_loc('gross_revenue_usd') + 1,
    'discount_amount_usd',
    ecommerce_df.eval('gross_revenue_usd * discount_percent / 100')
)

# Shipping cost as % of revenue
ecommerce_df.insert(
    ecommerce_df.columns.get_loc('shipping_cost_usd') + 1,
    'shipping_cost_percent',
    ecommerce_df.eval('shipping_cost_usd / revenue_usd * 100')
)

## 3. Discount Features

In [15]:
# Discount tier (adjusted to actual data range 0-25%)
ecommerce_df.insert(
    ecommerce_df.columns.get_loc('discount_percent') + 1,
    'discount_tier',
    pd.cut(
        ecommerce_df['discount_percent'],
        bins=[-1, 0, 10, 20, 25],
        labels=['No Discount', 'Low Discount', 'Medium Discount', 'High Discount']
    )
)

# Binary flag
ecommerce_df.insert(
    ecommerce_df.columns.get_loc('discount_tier') + 1,
    'is_discounted',
    np.where(
        ecommerce_df.eval('discount_percent > 0'),
        "Yes",
        "No"
    )
)

## 4. Customer Features

In [16]:
# Age group
ecommerce_df.insert(
    ecommerce_df.columns.get_loc('age') + 1,
    'age_group',
    pd.cut(
        ecommerce_df['age'],
        bins=[0, 24, 34, 44, 54, 64, 100],
        labels=['18-24', '25-34', '35-44', '45-54', '55-64', '65+']
    )
)

# Loyalty tier
ecommerce_df.insert(
    ecommerce_df.columns.get_loc('customer_loyalty_score') + 1,
    'loyalty_tier',
    pd.cut(
        ecommerce_df['customer_loyalty_score'],
        bins=[-1, 30, 70, 100],
        labels=['Low', 'Medium', 'High']
    )
)

## 5. Product & Order Features

In [17]:
# Price tier
ecommerce_df.insert(
    ecommerce_df.columns.get_loc('unit_price_usd') + 1,
    'price_tier',
    pd.cut(
        ecommerce_df['unit_price_usd'],
        bins=[0, 50, 100, 200, np.inf],
        labels=['Budget', 'Mid-Range', 'Premium', 'Luxury']
    )
)

# Quantity / basket size
ecommerce_df.insert(
    ecommerce_df.columns.get_loc('quantity') + 1,
    'quantity_segment',
    pd.cut(
        ecommerce_df['quantity'],
        bins=[0, 1, 3, 5],
        labels=['Single', 'Small Basket', 'Bulk']
    )
)

# Order value segment
ecommerce_df.insert(
    ecommerce_df.columns.get_loc('revenue_usd') + 1,
    'order_value_segment',
    pd.cut(
        ecommerce_df['revenue_usd'],
        bins=[0, 100, 300, 600, np.inf],
        labels=['Low', 'Medium', 'High', 'Very High']
    )
)

## 6. Logistics Features

In [18]:
ecommerce_df.insert(
    ecommerce_df.columns.get_loc('delivery_days') + 1,
    'delivery_speed',
    pd.cut(
        ecommerce_df['delivery_days'],
        bins=[0, 3, 7, 15],
        labels=['Fast', 'Standard', 'Slow']
    )
)
ecommerce_df.insert(
    ecommerce_df.columns.get_loc('shipping_country') + 1,
    'shipment_type',
    np.where(
        ecommerce_df['country'] == ecommerce_df['shipping_country'],
        "Domestic",
        "International"
    )
)

## 7. Behavior & Risk Features

In [19]:
# Engagement level
ecommerce_df.insert(
    ecommerce_df.columns.get_loc('session_duration_min') + 1,
    'engagement_level',
    pd.cut(
        ecommerce_df['session_duration_min'],
        bins=[-1, 15, 40, 100],
        labels=['Low', 'Medium', 'High']
    )
)

# Fraud risk level
ecommerce_df.insert(
    ecommerce_df.columns.get_loc('fraud_risk_score') + 1,
    'fraud_risk_level',
    pd.cut(
        ecommerce_df['fraud_risk_score'],
        bins=[-1, 30, 70, 100],
        labels=['Low', 'Medium', 'High']
    )
)

## 8. Final Check & Save

In [20]:
# Quick look at the engineered dataset
ecommerce_df.sample(10)

,order_date,order_year,order_month,order_day_of_week,part_day,is_weekend,customer_name,gender,age,age_group,customer_segment,country,order_status,category,sub_category,unit_price_usd,price_tier,quantity,quantity_segment,gross_revenue_usd,discount_amount_usd,discount_percent,discount_tier,is_discounted,revenue_usd,order_value_segment,profit_usd,profit_margin_percent,payment_method,shipping_method,shipping_cost_usd,shipping_cost_percent,delivery_days,delivery_speed,shipping_country,shipment_type,rating,customer_loyalty_score,loyalty_tier,coupon_used,session_duration_min,engagement_level,pages_visited,abandoned_cart_before,fraud_risk_score,fraud_risk_level,device_type,campaign_source,traffic_source
750958,2025-04-04 10:22:04.951013,2025,4,Fri,Morning,No,Jessica Cooper,Female,36,35-44,Regular,United States,Completed,Sports,Outdoor,156.76,Premium,1,Single,156.76,0.00,0,No Discount,No,156.76,Medium,77.89,49.69,Bank Transfer,Express,4.01,2.56,12,Slow,United States,Domestic,2,93.30,High,No,12.20,Low,6,Yes,39.70,Medium,Tablet,Organic,Direct
905699,2024-03-18 10:21:08.081929,2024,3,Mon,Morning,No,Dustin Sanders,Male,65,65+,Regular,France,Completed,Sports,Accessories,44.11,Budget,5,Bulk,220.55,11.03,5,Low Discount,Yes,209.52,Medium,81.27,38.79,Credit Card,Next Day,13.59,6.49,10,Slow,France,Domestic,4,43.10,Medium,Yes,43.00,High,10,Yes,47.90,Medium,Tablet,Email,Direct
209592,2025-10-30 22:31:18.497470,2025,10,Thu,Evening,No,Jillian Ward,Female,68,65+,Regular,United States,Completed,Electronics,Audio,115.15,Premium,3,Small Basket,345.45,0.00,0,No Discount,No,345.45,High,141.69,41.02,PayPal,Next Day,3.78,1.09,12,Slow,United States,Domestic,5,77.80,High,No,18.20,Medium,9,Yes,23.80,Low,Mobile,Instagram,Social
636695,2025-05-08 19:17:32.800130,2025,5,Thu,Evening,No,Raven Hansen,Female,38,35-44,Premium,United States,Completed,Sports,Outdoor,109.00,Premium,5,Bulk,545.00,0.00,0,No Discount,No,545.00,High,167.80,30.79,Credit Card,Economy,10.97,2.01,10,Slow,United States,Domestic,3,38.00,Medium,No,33.50,Medium,11,Yes,37.00,Medium,Tablet,Google Ads,Email
69503,2024-04-01 11:45:20.922704,2024,4,Mon,Morning,No,Deanna Zamora,Female,71,65+,Regular,Netherlands,Completed,Sports,Gym Equipment,27.40,Budget,3,Small Basket,82.20,12.33,15,Medium Discount,Yes,69.87,Low,29.76,42.59,Bank Transfer,Express,11.48,16.43,5,Standard,Netherlands,Domestic,1,47.50,Medium,Yes,52.50,High,11,No,28.50,Low,Mobile,Affiliate,Referral
983601,2024-11-20 13:36:08.878135,2024,11,Wed,Afternoon,No,Mary Peters,Female,50,45-54,Regular,Canada,Processing,Health,Medical Devices,92.46,Mid-Range,3,Small Basket,277.38,0.00,0,No Discount,No,277.38,Medium,134.10,48.35,PayPal,Standard,0.05,0.02,6,Standard,Canada,Domestic,4,93.70,High,No,14.50,Low,20,Yes,0.40,Low,Tablet,Email,Search
714128,2025-02-19 23:25:00.281679,2025,2,Wed,Evening,No,Nancy Martinez,Female,34,25-34,VIP,Australia,Pending,Clothing,Mens Wear,131.96,Premium,3,Small Basket,395.88,19.79,5,Low Discount,Yes,376.09,High,184.00,48.92,Apple Pay,Next Day,18.52,4.92,5,Standard,Australia,Domestic,3,20.10,Low,Yes,6.00,Low,16,No,92.40,High,Mobile,Google Ads,Email
361486,2024-04-02 22:59:17.762511,2024,4,Tue,Evening,No,Jennifer Frank,Female,59,55-64,Regular,United Kingdom,Returned,Sports,Accessories,122.98,Premium,3,Small Basket,368.94,36.89,10,Low Discount,Yes,332.05,High,123.34,37.15,Debit Card,Express,12.72,3.83,4,Standard,United Kingdom,Domestic,5,62.10,Medium,Yes,42.80,High,3,No,53.60,Medium,Mobile,Email,Search
282817,2025-06-02 05:21:22.934173,2025,6,Mon,Night,No,Patrick Young,Male,23,18-24,Regular,Germany,Completed,Sports,Gym Equipment,89.73,Mid-Range,3,Small Basket,269.19,26.92,10,Low Discount,Yes,242.27,Medium,81.83,33.78,Debit Card,Standard,23.58,9.73,5,Standard,Germany,Domestic,1,84.20,High,Yes,43.40,High,7,No,16.90,Low,Desktop,Google Ads,Direct
219601,2025-01-30 10:51:08.257314,2025,1,Thu,Morning,No,Todd Dorsey,Male,53,45-54,Premium,Australia,Pending,Health,Personal Care,25.90,Budget,1,Single,25.90,0.00,0,No Discount,No,25.90,Low,1

In [21]:
# Save the processed dataset
ecommerce_df.to_csv("dataset/processed/ecommerce_processed.csv", index=False)